In [2]:
import pandas as pd
from glob import glob

def load_results(fn_prefix):
    rm_fns = glob(f"results/{fn_prefix}_supervised_rm_seed*.jsonl")
    rm_df = pd.concat([pd.read_json(fn, lines=True) for fn in rm_fns])
    dpo_fns = glob(f"results/{fn_prefix}_supervised_dpo_seed*.jsonl")
    dpo_df = pd.concat([pd.read_json(fn, lines=True) for fn in dpo_fns])
    clf_fns = glob(f"results/{fn_prefix}_supervised_clf_seed*.jsonl")
    clf_df = pd.concat([pd.read_json(fn, lines=True) for fn in clf_fns])
    unsupervised_fns = glob(f"results/{fn_prefix}_unsupervised_seed*.jsonl")
    df = pd.concat([pd.read_json(fn, lines=True) for fn in unsupervised_fns])
    df["bt_rm"] = rm_df["bt_rm"].tolist()
    df["dpo_aligned"] = dpo_df["dpo_aligned"].tolist()
    df["clf"] = clf_df["clf"].tolist()
    return df

enzh_df = load_results("synthetic_enzh")
ende_df = load_results("synthetic_ende")
enfr_df = load_results("synthetic_enfr")
df = pd.concat([enzh_df, ende_df, enfr_df])

In [4]:
from sklearn.metrics import roc_auc_score, roc_curve

def compute_metrics(labels, scores):
    roc_auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    accuracy = 0
    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        acc = (preds == labels).mean()
        accuracy = max(accuracy, acc)
    return roc_auc, accuracy


ROW_NAMES = {
    "dpo_aligned": "DPO (Log-likelihood)",
    "bt_rm": "Bradley-Terry RM",
    "clf": "XLM-RoBERTa",
    
    "log_lklh_positive": "Log-likelihood",
    "negative_entropy_positive": "Entropy",
    "fast_detectgpt_positive": "Fast-DetectGPT",
    "mahalanobis_distance_positive": "Maha. Distance",
    "relative_mahalanobis_distance_positive": "Relative Maha. Dist.",
    "trajectory_volatility_positive": "Trajectory Volatility",

    "log_lklh_negative": "Log-likelihood",
    "negative_entropy_negative": "Entropy",
    "fast_detectgpt_negative": "Fast-DetectGPT",
    "mahalanobis_distance_negative": "Maha. Distance",
    "relative_mahalanobis_distance_negative": "Relative Maha. Dist.",
    "trajectory_volatility_negative": "Trajectory Volatility",

    "likelihood_ratios": "\\textbf{Translationese Index}"
}

def generate_cross_domain_results(df):
    enzh = df[df.file_path.str.contains("enzh")]
    enxx = df[~df.file_path.str.contains("enzh")]
    domains = {
        "qwen_ot": enzh[enzh.file_path.str.contains("oliver_twist_qwen")],
        "qwen_novel": enzh[(~enzh.file_path.str.contains("coca")&~enzh.file_path.str.contains("oliver_twist_qwen"))],
        "qwen_coca": enzh[(enzh.file_path.str.contains("coca")&enzh.file_path.str.contains("qwen"))],
        "llama_ot": enzh[enzh.file_path.str.contains("oliver_twist_llama")],
        "llama_novel": enzh[(~enzh.file_path.str.contains("coca")&~enzh.file_path.str.contains("oliver_twist_llama"))],
        "llama_coca": enzh[(enzh.file_path.str.contains("coca")&enzh.file_path.str.contains("llama"))],
        "ende": enxx[enxx.file_path.str.contains("ende")],
        "enfr": enxx[enxx.file_path.str.contains("enfr")],
    }
    for method in ROW_NAMES:
        row_latex_str = ROW_NAMES[method]
        for domain, data in domains.items():
            roc_auc, accuracy = compute_metrics(data.label, data[method])
            if method == "clf":
                accuracy = (data.label==data[method]).mean()
                row_latex_str += f" & ${accuracy*100:.1f}"
            else:
                row_latex_str += f" & ${accuracy*100:.1f}" + "_{~" + f"{roc_auc*100:.1f}" + "}$"
        row_latex_str += " \\\\"
        print(row_latex_str)

generate_cross_domain_results(df)

DPO (Log-likelihood) & $89.2_{~95.8}$ & $86.9_{~94.5}$ & $82.3_{~89.5}$ & $89.2_{~95.5}$ & $86.8_{~94.4}$ & $80.5_{~88.2}$ & $77.9_{~85.6}$ & $82.6_{~89.7}$ \\
Bradley-Terry RM & $94.7_{~98.4}$ & $87.6_{~94.2}$ & $87.0_{~93.9}$ & $87.3_{~94.3}$ & $89.9_{~95.6}$ & $87.2_{~93.8}$ & $72.9_{~79.8}$ & $76.5_{~84.6}$ \\
XLM-RoBERTa & $95.0 & $83.6 & $82.1 & $81.8 & $87.9 & $68.9 & $56.8 & $60.7 \\
Log-likelihood & $80.2_{~87.1}$ & $79.3_{~86.0}$ & $79.0_{~86.1}$ & $80.2_{~87.0}$ & $77.6_{~84.8}$ & $76.9_{~84.4}$ & $71.0_{~77.7}$ & $75.3_{~81.3}$ \\
Entropy & $73.8_{~77.0}$ & $64.9_{~71.0}$ & $69.1_{~73.7}$ & $71.7_{~74.5}$ & $65.4_{~71.5}$ & $71.1_{~78.0}$ & $60.4_{~63.9}$ & $61.1_{~65.4}$ \\
Fast-DetectGPT & $74.3_{~80.8}$ & $72.2_{~78.9}$ & $73.2_{~80.5}$ & $72.0_{~79.3}$ & $72.0_{~79.7}$ & $68.6_{~75.0}$ & $70.2_{~76.7}$ & $76.0_{~83.6}$ \\
Maha. Distance & $55.2_{~54.3}$ & $52.4_{~51.1}$ & $50.2_{~47.3}$ & $54.8_{~53.7}$ & $52.7_{~51.3}$ & $50.3_{~47.6}$ & $51.7_{~50.9}$ & $51.0_{~47.7}$

In [1]:
import pandas as pd

pointwise = pd.read_json("data/wild/pointwise.jsonl", lines=True)
pairwise = pd.read_json("data/wild/pairwise.jsonl", lines=True)

In [ ]:
from scipy.stats import pearsonr
import numpy as np


def compute_pearsonr(scores):
    return pearsonr(pointwise.mean_human_score, scores).statistic

def compute_accuracy(scores_A, scores_B):
    predictions = []
    for score_A, score_B in zip(scores_A, scores_B):
        prediction = "A" if score_A > score_B else "B"
        predictions.append(prediction)
    pairwise["correct"] = np.array(predictions) == pairwise.majority_vote
    return pairwise.groupby("agreement_cnt").aggregate(
        {"correct": "mean"}
    ).to_dict()["correct"]

def convert_pairwise_to_pointwise(df, k):
    df["source"] = df.messages_for_positive_model.apply(
        lambda x: x[0]["content"].split("\n")[-1]
    )
    df["translation"] = df.messages_for_positive_model.apply(
        lambda x: x[1]["content"]
    )
    scores = []
    for i, row in pointwise.iterrows():
        source, translation = row.source, row.translation
        data = df[(df.source == source) & (df.translation == translation)]
        if len(data) == 0:
            print(f"{source} {translation} not found")
        score = data[k].tolist()[0]
        scores.append(score)
    return scores

def convert_pointwise_to_pairwise(df, k):
    df["source"] = df.messages_for_positive_model.apply(
        lambda x: x[0]["content"].split("\n")[-1]
    )
    df["translation"] = df.messages_for_positive_model.apply(
        lambda x: x[1]["content"]
    )
    scores_A, scores_B = [], []
    for i, row in pairwise.iterrows():
        source, translation_A, translation_B = row.source, row.translation_A, row.translation_B
        data_A = df[(df.source == source) & (df.translation == translation_A)]
        score_A = data_A[k].tolist()[0]
        scores_A.append(score_A)
        data_B = df[(df.source == source) & (df.translation == translation_B)]
        score_B = data_B[k].tolist()[0]
        scores_B.append(score_B)
    return scores_A, scores_B

def print_results(results, row_name):
    row_str = row_name
    for k, v in results.items():
        row_str += f" & {v*100:.1f}"
    row_str += " \\\\"
    print(row_str)

n_sample=1000

df = pd.read_json(f"results/wild_t_index_sample{n_sample}_unpaired.jsonl", lines=True)
scores_A, scores_B = convert_pointwise_to_pairwise(df, "llr")
t_index_results = compute_accuracy(scores_A, scores_B)
t_index_results["r"] = float(compute_pearsonr(df.llr))
row_name = f"~~~~w/ unpaired. (1k)"
print_results(t_index_results, row_name)

df = pd.read_json(f"results/wild_t_index_sample{n_sample}_onedomain.jsonl", lines=True)
scores_A, scores_B = convert_pointwise_to_pairwise(df, "llr")
t_index_results = compute_accuracy(scores_A, scores_B)
t_index_results["r"] = float(compute_pearsonr(df.llr))
row_name = f"~~~~w/ single-dom. (1k)"
print_results(t_index_results, row_name)

for n_sample in [1000, 3000, 5000]:
    df = pd.read_json(f"results/wild_t_index_sample{n_sample}.jsonl", lines=True)
    scores_A, scores_B = convert_pointwise_to_pairwise(df, "llr")
    t_index_results = compute_accuracy(scores_A, scores_B)
    t_index_results["r"] = float(compute_pearsonr(df.llr))
    n = str(n_sample)[0]
    row_name = f"~~~~w/ mixed-dom. ({n}k)"
    print_results(t_index_results, row_name)


n_sample=5000
df = pd.read_json(f"results/wild_rm_samples{n_sample}.jsonl", lines=True)
scores_A = df[df.label==1]["bt_rm"].tolist()
scores_B = df[df.label==0]["bt_rm"].tolist()
bt_rm_results = compute_accuracy(scores_A, scores_B)
bt_rm = convert_pairwise_to_pointwise(df, "bt_rm")
bt_rm_results["r"] = float(compute_pearsonr(bt_rm))
print_results(bt_rm_results, "BT RM")

df = pd.read_json(f"results/wild_dpo_samples{n_sample}.jsonl", lines=True)
scores_A = df[df.label==1]["dpo_aligned"].tolist()
scores_B = df[df.label==0]["dpo_aligned"].tolist()
dpo_results = compute_accuracy(scores_A, scores_B)
dpo = convert_pairwise_to_pointwise(df, "dpo_aligned")
dpo_results["r"] = float(compute_pearsonr(dpo))
print_results(dpo_results, "DPO")

~~~~w/ unpaired. (1k) & 64.4 & 66.7 & 82.1 & 34.8 \\
~~~~w/ single-dom. (1k) & 68.9 & 74.2 & 84.6 & 34.6 \\
~~~~w/ mixed-dom. (1k) & 68.9 & 77.3 & 82.1 & 32.0 \\
~~~~w/ mixed-dom. (3k) & 55.6 & 74.2 & 74.4 & 39.2 \\
~~~~w/ mixed-dom. (5k) & 57.8 & 74.2 & 84.6 & 41.8 \\
BT RM & 57.8 & 72.7 & 76.9 & 40.7 \\
DPO & 62.2 & 66.7 & 76.9 & 19.7 \\
